In [ ]:
# Import des librairies

import pandas as pd
import numpy as np
from datetime import datetime

# Charger les DF des différents pages

DF : projects_plans, actual_duration, project_type, locations, country_profiles  
- Vérifier que les headers sont OK  
- afficher les types de données  
- afficher actual_duration, projects_plans


In [ ]:
fichier = 'data\donnees_sanitoral.xlsx'

dfs = pd.read_excel(fichier, sheet_name=None)

for nom_feuille, df in dfs.items():
    globals()[f"df_{nom_feuille}"] = df
    df.info()

In [ ]:
df_projects_plans = pd.read_excel(fichier, sheet_name="Projects_plans", header=3)
df_projects_plans.head()

,1,Phase 1 - Planning,2018-01-16 00:00:00,196,50000,10
0,1,Phase 2 - Initiation,2018-01-31,15,100000,17
1,1,Phase 3 - Implementation,2018-04-17,197,150000,26
2,1,Phase 4 - Manufacturing,2018-10-10,61,450000,23
3,2,Phase 1 - Planning,2018-01-30,16,100000,6
4,2,Phase 2 - Initiation,2018-04-01,109,200000,27


In [ ]:
df.head()

,1,Phase 1 - Planning,2018-01-16 00:00:00,196,50000,10
0,1,Phase 2 - Initiation,2018-01-31,15,100000,17
1,1,Phase 3 - Implementation,2018-04-17,197,150000,26
2,1,Phase 4 - Manufacturing,2018-10-10,61,450000,23
3,2,Phase 1 - Planning,2018-01-30,16,100000,6
4,2,Phase 2 - Initiation,2018-04-01,109,200000,27


# Jointures

Joindre projects_plans et actual_duration dans un df, par Project ID et Phase  
Attention : vérifier la jointure


In [ ]:
# jointures
df_merge = pd.merge(projects_plans, actual_duration, 
              left_on=['Project ID', 'Phase'], 
              right_on=['Project', 'Phase'], 
              how='outer', 
              indicator = True)
df_merge

KeyError: 'Phase'

In [ ]:
df_merge.value_counts()

# Calculs

Ajouter dans le DF :  
- le retard (j) : Delay = Actual_Duration - Planned_Duration
- Is_Late : 1 = retard, 0 = pas de retard
- Start_Month, Start_Quarter, Start_Year

In [ ]:
# Calcul du retard (en jours)
df_merge['Delay'] = df_merge['Actual_Duration'] - df_merge['Planned_Duration']
df_merge['Is_Late'] = (df_merge['Delay'] > 0).astype(int)  # Variable binaire : 1 = retard, 0 = pas de retard

# Degré de retard (catégories)
df_merge['Delay_Category'] = pd.cut(df_merge['Delay'], 
                               bins=[-float('inf'), -1, 0, 30, 90, float('inf')],
                               labels=['En avance', 'À l\'heure', 'Petit retard (<1mois)', 
                                      'Retard moyen (1-3mois)', 'Grand retard (>3mois)'])
df_merge

In [ ]:
# Mois de début (saisonnalité)
df_merge['Start_Month'] = pd.to_datetime(df_merge['Start Date']).dt.month
df_merge['Start_Quarter'] = pd.to_datetime(df_merge['Start Date']).dt.quarter
df_merge['Start_Year'] = pd.to_datetime(df_merge['Start Date']).dt.year

# Charge de travail initiale
df_merge['Planned_Intensity'] = df_merge['Planned_Delivrable'] / df_merge['Planned_Duration'] * 30  # livrables/mois
df_merge

# Aggrégations

Récupérer par projet (Project ID) :  
- somme Planned_Duration  
- somme Planned_Cost
- somme Planned_Delivrable
- nombre de Phase
- somme Delay
- max Is_Late
- Start_Month
- Start_Year



In [ ]:
#df_proj = df.groupby()...
project_features = df_merge.groupby('Project ID').agg({
    'Planned_Duration': 'sum',
    'Planned_Cost': 'sum',
    'Planned_Delivrable': 'sum',
    'Phase': 'count',
    'Delay': 'sum',
    'Is_Late': 'max',
    'Start_Month': 'first',
    'Start_Year': 'first'
}).rename(columns={
    'Planned_Duration': 'Total_Planned_Duration',
    'Planned_Cost': 'Total_Planned_Cost',
    'Planned_Delivrable': 'Total_Planned_Delivrables',
    'Phase': 'Nb_Phases',
    'Delay': 'Total_Delay',
    'Is_Late': 'Project_Had_Late_Phase'
})

project_features

In [ ]:
# Fusion avec le type de projet
df_merge = pd.merge(df_merge, project_type, on='Project ID', how='left')

# Fusion avec la localisation
df_merge = pd.merge(df_merge, locations, on='Project ID', how='left')
df_merge = pd.merge(df_merge, country_profiles, on='Country', how='left')

df_merge

In [ ]:
(projects_plans['Planned_Duration']>190).value_counts()

# Graphique histogram

Histogramme des Planned_Duration

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Figure 1: Histogramme de base
plt.figure(figsize=(12, 6))

plt.hist(df_merge['Delay'], 
         #bins=30, 
         color='steelblue', 
         edgecolor='white',
         alpha=0.7)

plt.title('Distribution des Actual_Duration-Planned_Duration (Delay)', fontsize=14, fontweight='bold')

plt.show()
